In [ ]:
import json
import os
import random

from datasets import Dataset, DatasetDict, Features, Value, load_from_disk
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [6]:
p = Path("../../data/OA-mine")
annotated_files = os.listdir(p / "annotations")
annotated_categories = [file.removesuffix('.jsonl') for file in annotated_files]
print(f"Annotated categories: {annotated_categories}\n")
with open(p / "annotations" / annotated_files[0]) as f:
    content_sample = "".join(f.readlines()[:3])
    print(f'Annotated contents: \n{content_sample}')

with open(p / "raw" / annotated_files[0]) as f:
    content_sample = "".join(f.readlines()[:3])
    print(f"Raw contents: \n{content_sample}")
    

Annotated categories: ['breakfast_cereal', 'safety_mask', 'tea', 'conditioner', 'sunscreen', 'toothbrush', 'laundry_detergent', 'shoes', 'coffee', 'vitamin']

Annotated contents: 
{"asin": "B074H64BGT", "title": "365 Everyday Value, Organic Raisin Granola, 17 oz", "entities": [{"startOffset": 0, "endOffset": 18, "label": "Brand", "value": "365 Everyday Value"}, {"startOffset": 20, "endOffset": 27, "label": "Specialty", "value": "Organic"}, {"startOffset": 28, "endOffset": 42, "label": "Flavor", "value": "Raisin Granola"}, {"startOffset": 44, "endOffset": 49, "label": "Net content", "value": "17 oz"}]}
{"asin": "B07FJ48RXG", "title": "Powerful Overnight Oats Oatmeal Cup, High Protein, Whole Grain, Kosher, Natural Ingredients, 20g Protein, Blueberry and Vanilla, 6 Pack", "entities": [{"startOffset": 0, "endOffset": 23, "label": "Brand", "value": "Powerful Overnight Oats"}, {"startOffset": 24, "endOffset": 35, "label": "Container", "value": "Oatmeal Cup"}, {"startOffset": 37, "endOffset":

In [7]:
def build_asin_index(jsonl_file):
    """
    Build an index of ASINs from a JSON Lines file.
    
    Args:
        jsonl_file (str): Path to the JSON Lines file
    
    Returns:
        dict: Dictionary mapping ASINs to their title and description
    """
    asin_index = {}
    
    with open(jsonl_file, 'r') as file:
        for line in file:
            try:
                # The file might have multiple JSON objects per line
                line_content = line.strip()
                
                # Handle case where multiple JSON objects are on the same line
                current_pos = 0
                while current_pos < len(line_content):
                    try:
                        # Try to find a valid JSON object
                        data = json.loads(line_content[current_pos:])
                        
                        # If successful, store in index and move to next line
                        if 'asin' in data:
                            asin_index[data['asin']] = {
                                'title': data.get('title', ''),
                                'description': data.get('description', '')
                            }
                        break
                    except json.JSONDecodeError as e:
                        # If error at specific position, try to find the end of current object
                        if current_pos + e.pos < len(line_content):
                            # Move past this character and try again
                            current_pos += e.pos + 1
                        else:
                            break
            except Exception:
                # Skip problematic lines
                continue
    
    return asin_index

In [8]:
def search_multiple_asins(jsonl_file, target_asins):
    """
    Search for multiple ASINs in a JSON Lines file.
    
    Args:
        jsonl_file (str): Path to the JSON Lines file
        target_asins (list): List of ASINs to search for
    
    Returns:
        dict: Dictionary mapping ASINs to their details (if found)
    """
    # Build index once
    asin_index = build_asin_index(jsonl_file)
    
    # Get results for all target ASINs
    results = {}
    for asin in target_asins:
        if asin in asin_index:
            results[asin] = asin_index[asin]
    
    return results

def clean_category_name(filename):
    """Convert filename to clean category name"""
    name = filename.replace('.jsonl', '')
    name = name.replace('_', ' ')
    return name.title()

In [9]:
for file in annotated_files:
    asins = []
    with open(p / "annotations" / file, "r") as ann_file:
        for line in ann_file.readlines():
            try:
                product = json.loads(line)
            except Exception as e:
                print(f"Caught error: {e}")
            asins.append(product['asin'])

        matches = search_multiple_asins((p / "raw" / file), asins)
        if matches:
            print(f"Found {len(matches)} matches for {len(asins)} products in {clean_category_name(file)}:")
            # for asin, details in matches.items():
            #     print(f"\nASIN: {asin}")
            #     print(f"Title: {details['title']}")
            #     print(f"Description: {details['description']}")
        else:
            print("No matches found")

Found 197 matches for 197 products in Breakfast Cereal:
Found 198 matches for 198 products in Safety Mask:
Found 197 matches for 197 products in Tea:
Found 178 matches for 178 products in Conditioner:
Found 197 matches for 197 products in Sunscreen:
Found 197 matches for 197 products in Toothbrush:
Found 191 matches for 191 products in Laundry Detergent:
Found 189 matches for 189 products in Shoes:
Found 200 matches for 200 products in Coffee:
Found 199 matches for 199 products in Vitamin:


In [10]:
def merge_and_format_data(annotations_path, raw_path, files):
    """Merge raw and annotated data and format to target structure"""
    formatted_data = []
    
    for file in files:
        category = clean_category_name(file)
        print(f"Processing category: {category}")
        
        # Read annotated data
        annotated_products = {}
        with open(annotations_path / file, "r") as ann_file:
            for line in ann_file.readlines():
                try:
                    product = json.loads(line)
                    annotated_products[product['asin']] = product
                except Exception as e:
                    print(f"Error parsing annotated data: {e}")
        
        # Read raw data
        raw_products = {}
        with open(raw_path / file, "r") as raw_file:
            for line in raw_file.readlines():
                try:
                    product = json.loads(line)
                    raw_products[product['asin']] = product
                except Exception as e:
                    print(f"Error parsing raw data: {e}")
        
        # Format and merge
        for asin, annotated_product in annotated_products.items():
            # Create paragraphs array with title
            paragraphs = [
                {"text": annotated_product["title"], "source": "title"}
            ]
            
            # Add description if available in raw data
            if asin in raw_products and raw_products[asin].get("description"):
                paragraphs.append({
                    "text": raw_products[asin]["description"], 
                    "source": "description"
                })
            
            # Convert entities to labels
            labels = []
            for entity in annotated_product.get("entities", []):
                labels.append({
                    "key": entity["label"],
                    "value": entity["value"]
                })
            
            # Create formatted entry
            formatted_entry = {
                "id": asin,
                "category": category,
                "paragraphs": paragraphs,
                "labels": labels
            }
            
            formatted_data.append(formatted_entry)
    
    return formatted_data

In [11]:
def create_stratified_split(data, test_size=0.15, val_size=0.10):
    """Create stratified splits across all categories"""
    # Group by category for stratified sampling
    category_groups = {}
    for item in data:
        cat = item["category"]
        if cat not in category_groups:
            category_groups[cat] = []
        category_groups[cat].append(item)
    
    train_data, test_data, val_data = [], [], []
    
    for cat, items in category_groups.items():
        # First split off the test set
        train_val, test = train_test_split(items, test_size=test_size, random_state=42)
        
        # Then split the remaining data into train and validation
        adjusted_val_size = val_size / (1 - test_size)
        train, val = train_test_split(train_val, test_size=adjusted_val_size, random_state=42)
        
        train_data.extend(train)
        test_data.extend(test)
        val_data.extend(val)
    
    return {
        "train": train_data,
        "test": test_data,
        "validation": val_data
    }

In [12]:
def create_category_split(data, categories):
    """Create a split where specific categories are held out"""
    # Select categories for validation and test
    all_categories = list(set(item["category"] for item in data))
    if not categories:
        random.seed(42)
        random.shuffle(all_categories)
        val_category, test_category = all_categories[:2]
    else:
        val_category, test_category = categories
    
    print(f"Using '{val_category}' for validation and '{test_category}' for test")
    
    # Split based on categories
    train_data, test_data, val_data = [], [], []
    
    for item in data:
        if item["category"] == val_category:
            val_data.append(item)
        elif item["category"] == test_category:
            test_data.append(item)
        else:
            train_data.append(item)
    
    return {
        "train": train_data,
        "test": test_data,
        "validation": val_data
    }

In [13]:
# Debug function to validate data structure
def debug_data_format(data):
    """Print a sample entry to verify structure"""
    if not data:
        print("No data found!")
        return
    
    sample = data[0]
    print("\nSample data entry BEFORE dataset creation:")
    print(f"ID: {sample['id']}")
    print(f"Category: {sample['category']}")
    print(f"Paragraphs type: {type(sample['paragraphs'])}")
    print(f"Paragraphs sample: {sample['paragraphs'][0]}")
    print(f"Labels type: {type(sample['labels'])}")
    print(f"Labels sample: {sample['labels'][0] if sample['labels'] else 'None'}")

In [14]:
def create_dataset_dict(split_data):
    """Convert split data into a DatasetDict with correct structure preservation"""
    
    # Define features that preserve the structure
    features = Features({
        "id": Value("string"),
        "category": Value("string"),
        "paragraphs": [
            {
                "text": Value("string"),
                "source": Value("string")
            }
        ],
        "attributes": [
            {
                "key": Value("string"),
                "evidences": [
                    {
                        "value": Value("string"),
                        "pid": Value("int64"),
                        "begin": Value("int64"),
                        "end": Value("int64")
                    }
                ]
            }
        ],
        "labels": [
            {
                "key": Value("string"),
                "value": Value("string")
            }
        ]
    })
    
    # Add attributes field to data if not present
    for item in split_data["train"] + split_data["test"] + split_data["validation"]:
        if "attributes" not in item:
            item["attributes"] = []
    
    dataset_dict = DatasetDict()
    for split_name, data in split_data.items():
        dataset_dict[split_name] = Dataset.from_list(data, features=features)
    
    return dataset_dict

In [15]:
# Main script to create datasets
def create_oa_datasets(base_path, output_dir, debug=False):
    p = Path(base_path)
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all annotation files
    annotated_files = [f for f in os.listdir(p / "annotations") if f.endswith('.jsonl')]
    
    # Merge and format the data
    formatted_data = merge_and_format_data(p / "annotations", p / "raw", annotated_files)
    print(f"Formatted {len(formatted_data)} products total")
    
    # Debug: Check data structure
    if debug:
        debug_data_format(formatted_data)
    
    # Create stratified split dataset
    stratified_splits = create_stratified_split(formatted_data)
    stratified_dataset = create_dataset_dict(stratified_splits)
    
    # Create category-based split dataset
    category_splits = create_category_split(formatted_data, categories=None)
    category_dataset = create_dataset_dict(category_splits)
    
    if debug: # Check dataset structure
        print("\nSample from dataset AFTER creation:")
        sample = stratified_dataset["train"][0]
        print(f"Sample paragraphs: {sample['paragraphs']}")
        print(f"Sample labels: {sample['labels']}")
    
    # Save datasets
    stratified_dataset.save_to_disk(os.path.join(output_dir, "stratified_split"))
    category_dataset.save_to_disk(os.path.join(output_dir, "category_split"))
    
    # Print summary
    print("\nDataset Summary:")
    print("\nStratified Split:")
    for split, dataset in stratified_dataset.items():
        print(f"  {split}: {len(dataset)} examples")
    
    print("\nCategory Split:")
    for split, dataset in category_dataset.items():
        print(f"  {split}: {len(dataset)} examples")
    
    return stratified_dataset, category_dataset

In [16]:
# Set paths
base_path = p 
output_dir = "../../data/OA-mine/dataset"  

# Create the datasets
stratified_dataset, category_dataset = create_oa_datasets(base_path, output_dir, False)

Processing category: Breakfast Cereal
Processing category: Safety Mask
Processing category: Tea


Processing category: Conditioner
Processing category: Sunscreen
Processing category: Toothbrush
Processing category: Laundry Detergent
Processing category: Shoes
Processing category: Coffee
Processing category: Vitamin
Formatted 1943 products total
Using 'Breakfast Cereal' for validation and 'Laundry Detergent' for test


Saving the dataset (1/1 shards): 100%|██████████| 197/197 [00:00<00:00, 8948.79 examples/s]


Dataset Summary:

Stratified Split:
  train: 1450 examples
  test: 295 examples
  validation: 198 examples

Category Split:
  train: 1555 examples
  test: 191 examples
  validation: 197 examples


In [17]:
print(stratified_dataset['train'][0])

mave = load_from_disk('/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/data/MAVE/dataset')
print(mave['train'][0])

{'id': 'B00Y0UWNTC', 'category': 'Breakfast Cereal', 'paragraphs': [{'text': 'Cascadian Farm Organic Honey Crunch Oat Cereal, 13.5 oz', 'source': 'title'}, {'text': 'Cascadian Farm Organic Honey Oat Cereal is loaded with whole grain oats, organic corn flakes, wheat flakes, and honey. Cereal you can feel good about feeding your loved ones, with enough for everyone!', 'source': 'description'}], 'labels': [{'key': 'Brand', 'value': 'Cascadian Farm'}, {'key': 'Specialty', 'value': 'Organic'}, {'key': 'Flavor', 'value': 'Honey Crunch'}, {'key': 'Sub brand', 'value': 'Oat Cereal'}, {'key': 'Net content', 'value': '13.5 oz'}], 'attributes': []}
{'id': '6342506256', 'category': 'Shorts', 'paragraphs': [{'text': "Gaok Men's Retro Cotton Multi-Pocket Camo Cargo Shorts", 'source': 'title'}, {'text': "Gaok men's pants are all made from cotton materials,which provide you soft,cosy and breathable skin touch. they are in fashionable design with multi cargo pocket,loose style.they are perfect for spor

In [ ]:
def create_length_limited_dataset(dataset_dict, max_tokens=896, output_dir="../../data/OA-mine/dataset/truncated_stratified_split"):
    """
    Create a version of the dataset where paragraphs are truncated to not exceed max_tokens.
    
    Args:
        dataset_dict: The original DatasetDict
        max_tokens: Maximum number of tokens allowed per example
        output_dir: Where to save the truncated dataset
    
    Returns:
        Truncated DatasetDict
    """
    
    # Initialize tokenizer (using BERT base as a reasonable default)
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
    
    def truncate_paragraphs(example):
        # Copy the example
        new_example = {k: v for k, v in example.items()}
        
        # Initialize token count
        total_tokens = 0
        truncated_paragraphs = []
        
        # Process each paragraph
        for p in example["paragraphs"]:
            # Tokenize the paragraph
            tokens = tokenizer.tokenize(p["text"])
            tokens_count = len(tokens)
            
            # If adding this paragraph would exceed limit
            if total_tokens + tokens_count > max_tokens:
                # Calculate how many tokens we can still add
                remaining_tokens = max_tokens - total_tokens
                if remaining_tokens > 0:
                    # Add a truncated version of the paragraph
                    truncated_text = tokenizer.convert_tokens_to_string(
                        tokens[:remaining_tokens]
                    )
                    truncated_paragraphs.append({
                        "text": truncated_text, 
                        "source": p["source"]
                    })
                break
            else:
                # Add the full paragraph
                truncated_paragraphs.append(p)
                total_tokens += tokens_count
        
        # Make sure we have at least one paragraph
        if not truncated_paragraphs and example["paragraphs"]:
            first_p = example["paragraphs"][0]
            tokens = tokenizer.tokenize(first_p["text"])
            truncated_text = tokenizer.convert_tokens_to_string(
                tokens[:max_tokens]
            )
            truncated_paragraphs.append({
                "text": truncated_text,
                "source": first_p["source"]
            })
            
        new_example["paragraphs"] = truncated_paragraphs
        return new_example
    
    # Create a new DatasetDict with truncated examples
    truncated_dict = dataset_dict.map(truncate_paragraphs)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Save the truncated dataset
    truncated_dict.save_to_disk(output_dir)
    
    # Print summary
    print("\nTruncated Dataset Summary:")
    for split, dataset in truncated_dict.items():
        print(f"  {split}: {len(dataset)} examples")
    
    return truncated_dict

In [19]:
# After creating the original datasets
truncated_dataset = create_length_limited_dataset(stratified_dataset)

# Print a comparison of an example before and after truncation
print("\nOriginal vs Truncated Example:")
example_id = stratified_dataset["train"][0]["id"]
original = next(ex for ex in stratified_dataset["train"] if ex["id"] == example_id)
truncated = next(ex for ex in truncated_dataset["train"] if ex["id"] == example_id)

# Count tokens in original and truncated
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
original_tokens = sum(len(tokenizer.tokenize(p["text"])) for p in original["paragraphs"])
truncated_tokens = sum(len(tokenizer.tokenize(p["text"])) for p in truncated["paragraphs"])

print(f"Original paragraphs count: {len(original['paragraphs'])}")
print(f"Original tokens: {original_tokens}")
print(f"Truncated paragraphs count: {len(truncated['paragraphs'])}")
print(f"Truncated tokens: {truncated_tokens}")

Saving the dataset (1/1 shards): 100%|██████████| 198/198 [00:00<00:00, 20144.38 examples/s]



Truncated Dataset Summary:
  train: 1450 examples
  test: 295 examples
  validation: 198 examples

Original vs Truncated Example:
Original paragraphs count: 2
Original tokens: 60
Truncated paragraphs count: 2
Truncated tokens: 60


/vol/csedu-nobackup/project/lzandbergen-storage/venv2/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
